# EuclidRL - Theorem Proving with Qwen2.5-Math (Colab)

This notebook trains theorem provers using:
1. **SFT (Supervised Fine-Tuning)** - Initial training on theorem-proof pairs
2. **GRPO (Group Relative Policy Optimization)** - RL training with Lean 4 feedback

**Steps:**
1. Connect to Colab GPU (Runtime → Change runtime type → T4 GPU)
2. Run each cell from top to bottom
3. Set your Hugging Face token in the first code cell
4. Choose SFT or GRPO training

**Repository:** [CornellDataScience/EuclidRL](https://github.com/CornellDataScience/EuclidRL)

In [ ]:
import os
# IMPORTANT: Set your Hugging Face token here (required for downloading models)
# Get your token from: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = ""  # ⚠️ PASTE YOUR TOKEN HERE

if not os.environ["HF_TOKEN"]:
    print("⚠️  WARNING: HF_TOKEN is empty! Set it above before continuing.")
else:
    print("✓ HF_TOKEN is set")

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone the repo from CornellDataScience/EuclidRL
# Using the feature branch with LeanDojo 4.20.0 fixes
REPO_URL = "https://github.com/CornellDataScience/EuclidRL.git"
BRANCH = "feature/deepseek-rl-sft-assets"
REPO_DIR = "EuclidRL"

import pathlib, subprocess
if not pathlib.Path(REPO_DIR).exists():
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL], check=True)
    print("✓ Repository cloned successfully")
else:
    print(f"✓ Repository already exists at {REPO_DIR}")

%cd {REPO_DIR}

In [ ]:
# Install Python dependencies
print("Installing dependencies...")
!pip install --upgrade pip
!pip install -r requirements.txt
print("\n✓ Dependencies installed")

In [ ]:
# Prepare training data (requires proof-bearing JSONL)
# Expected file: data/raw/deepseek_v1_train.jsonl (DeepSeek-Prover V1 with proofs)
# Upload it to this path, then rerun this cell.

import pathlib, subprocess, sys
raw_path = pathlib.Path('data/raw/deepseek_v1_train.jsonl')
raw_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_path.exists():
    print('⚠️  Missing proof dataset:', raw_path)
    print('Upload the DeepSeek-Prover V1 proof JSONL to this path, then rerun this cell.')
    print('\nAlternatively, you can use your own training data in JSONL format:')
    print('  {"formal_statement": "theorem ...", "proof": "by ..."}')
    sys.exit(1)

print('Found dataset at', raw_path)

print('\nConverting to SFT format...')
subprocess.run([
    sys.executable,
    'scripts/convert_deepseek_v15.py',
    '--inputs', str(raw_path),
    '--train-output', 'data/sft/train.jsonl',
    '--val-output', 'data/sft/val.jsonl',
], check=True)

print('\n✓ Training data ready!')
print('  - Train: data/sft/train.jsonl')
print('  - Val:   data/sft/val.jsonl')

## Part 1: SFT (Supervised Fine-Tuning)

Train the model on theorem-proof pairs to learn the basic proof structure.

This creates the initial checkpoint needed for GRPO training.

In [ ]:
# Run SFT training
# Make sure we are in the EuclidRL directory and have latest code
%cd /content/EuclidRL
!git pull origin feature/deepseek-rl-sft-assets
!python scripts/train_sft.py --config prover/configs/sft_default.yaml

In [ ]:
# Zip checkpoints for download
import pathlib
if pathlib.Path("checkpoints/sft").exists():
    !zip -r sft_checkpoints.zip checkpoints/sft
    print("✓ SFT checkpoints zipped!")
    print("📦 Download sft_checkpoints.zip from the Files panel (left sidebar)")
else:
    print("⚠️  No SFT checkpoints found. Training may not have completed successfully.")

## Part 2: GRPO (Group Relative Policy Optimization)

**Optional**: Further improve the model using reinforcement learning with Lean 4 feedback.

**Requirements:**
- SFT checkpoint from Part 1 (above)
- Training data with theorem statements (JSONL format)
- Lean 4 installation (for proof verification)

**What GRPO does:**
1. Generates multiple candidate proofs per theorem
2. Verifies each with Lean 4 (binary reward: 1 or 0)
3. Updates policy to favor higher-reward proofs within each group
4. No critic/value network needed (unlike PPO)

See [README_GRPO.md](README_GRPO.md) for details.

In [ ]:
# Install Lean 4 (required for proof verification)
# This may take a few minutes
print("Installing Lean 4...")
!curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y
!~/.elan/bin/elan default leanprover/lean4:stable
print("\n✓ Lean 4 installed")

In [ ]:
# Prepare GRPO training data
# Format: {"prompt": "theorem statement", "theorem": "TheoremName"}
#
# You can reuse the SFT data or create new rollouts
import pathlib
rollout_dir = pathlib.Path("data/rollouts")
rollout_dir.mkdir(parents=True, exist_ok=True)

# Option 1: Copy from SFT data
if pathlib.Path("data/sft/train.jsonl").exists():
    !cp data/sft/train.jsonl data/rollouts/train.jsonl
    print("✓ Copied SFT data to rollouts/train.jsonl")
else:
    print("⚠️  No SFT data found. Upload data/rollouts/train.jsonl manually.")

In [ ]:
# Run GRPO training
# Uses DeepSeek-Prover-V1.5 hyperparameters:
#   - learning_rate: 5e-6
#   - batch_size: 512 (reduced for Colab)
#   - group_size: 32 (reduced for Colab)
#   - kl_penalty: 0.02

%cd /content/EuclidRL
!python scripts/train_grpo.py \
    --config prover/configs/grpo_default.yaml \
    --batch-size 16 \
    --group-size 8

In [ ]:
# Zip GRPO checkpoints for download
import pathlib
if pathlib.Path("checkpoints/grpo").exists():
    !zip -r grpo_checkpoints.zip checkpoints/grpo
    print("✓ GRPO checkpoints zipped!")
    print("📦 Download grpo_checkpoints.zip from the Files panel (left sidebar)")
else:
    print("⚠️  No GRPO checkpoints found. Training may not have completed successfully.")